# Features for Churn Prediction

**Goal:** for every customer and snapshot date, build features using only data before the cutoff date.
**Input:** `sales.parquet`, `returns.parquet` and `labels.parquet`. **Output:** `features.parquet`.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

# Show up to 50 columns when printing a table
pd.set_option("display.max_columns", 50)

# One clean style for all charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

# Folder where the charts are saved (they will be used in the README)
fig_dir = Path("../reports/figures")
fig_dir.mkdir(parents=True, exist_ok=True)


def takeaway(text):
    # Show the takeaway under a chart. The numbers are computed from the data.
    display(Markdown("**Takeaway:** " + text))

In [4]:
# Load the cleaned tables and the labels from the previous notebooks
sales = pd.read_parquet("../data/processed/sales.parquet")
returns = pd.read_parquet("../data/processed/returns.parquet")
labels = pd.read_parquet("../data/processed/labels.parquet")

# Sort by date once, so "last value" lookups later are correct
sales = sales.sort_values("invoicedate").reset_index(drop=True)

# Revenue of each sales row = quantity x price
sales["line_total"] = sales["quantity"] * sales["price"]

# Value of each return row (quantity is negative in returns, so take the absolute value)
returns["return_value"] = (returns["quantity"] * returns["price"]).abs()

print("Sales:", sales.shape, "| Returns:", returns.shape, "| Labels:", labels.shape)

Sales: (775278, 10) | Returns: (16285, 10) | Labels: (36297, 4)


## Build an invoice-level table

In [5]:
# One row per invoice (one purchase = one invoice)
invoices = (
    sales.groupby(["customer_id", "invoice"])
    .agg(
        invoice_date=("invoicedate", "min"),
        revenue=("line_total", "sum"),
        lines=("stockcode", "size"),    # number of product lines in the invoice
        items=("quantity", "sum"),      # number of items in the invoice
    )
    .reset_index()
)

# The calendar day of each invoice (several invoices on one day count as one purchase day)
invoices["invoice_day"] = invoices["invoice_date"].dt.normalize()

print("Invoices:", f"{len(invoices):,}", "| Customers:", f"{invoices['customer_id'].nunique():,}")
labels.head()

Invoices: 36,480 | Customers: 5,851


,customer_id,cutoff,split,churn
0,12346,2010-06-14,train,0
1,12349,2010-06-14,train,1
2,12355,2010-06-14,train,1
3,12358,2010-06-14,train,1
4,12359,2010-06-14,train,0


## Build the features for one snapshot

In [6]:
def make_features(cutoff):
    cutoff = pd.Timestamp(cutoff)
    ids = labels.loc[labels["cutoff"] == cutoff, "customer_id"]

    # PAST ONLY: every table is cut at the cutoff date, then limited to the customers of this snapshot
    inv = invoices[(invoices["invoice_date"] < cutoff) & invoices["customer_id"].isin(ids)]
    sal = sales[(sales["invoicedate"] < cutoff) & sales["customer_id"].isin(ids)]
    ret = returns[(returns["invoicedate"] < cutoff) & returns["customer_id"].isin(ids)]

    # Basic numbers per customer from the invoice table
    g = inv.groupby("customer_id")
    feat = pd.DataFrame({
        "last_purchase": g["invoice_date"].max(),
        "first_purchase": g["invoice_date"].min(),
        "n_invoices": g.size(),
        "n_purchase_days": g["invoice_day"].nunique(),
        "monetary_total": g["revenue"].sum(),
        "median_invoice_value": g["revenue"].median(),
        "avg_items_per_invoice": g["items"].mean(),
        "avg_lines_per_invoice": g["lines"].mean(),
    })

    # Recent activity: last 90 and last 180 days before the cutoff
    for days in (90, 180):
        recent = inv[inv["invoice_date"] >= cutoff - pd.Timedelta(days=days)]
        rg = recent.groupby("customer_id")
        feat[f"n_invoices_{days}d"] = rg.size()
        feat[f"monetary_{days}d"] = rg["revenue"].sum()

    # Average number of days between two purchase days of the same customer
    purchase_days = inv[["customer_id", "invoice_day"]].drop_duplicates().sort_values(["customer_id", "invoice_day"])
    gaps = purchase_days.groupby("customer_id")["invoice_day"].diff().dt.days
    feat["avg_gap_days"] = gaps.groupby(purchase_days["customer_id"]).mean()

    # Number of different products, and the country of the last purchase
    feat["n_products"] = sal.groupby("customer_id")["stockcode"].nunique()
    feat["is_uk"] = (sal.groupby("customer_id")["country"].last() == "United Kingdom").astype(int)

    # Returns before the cutoff (customers without returns get 0)
    rg = ret.groupby("customer_id")
    feat["n_return_lines"] = rg.size()
    feat["return_value"] = rg["return_value"].sum()
    feat[["n_return_lines", "return_value"]] = feat[["n_return_lines", "return_value"]].fillna(0)
    feat[["n_invoices_90d", "monetary_90d", "n_invoices_180d", "monetary_180d"]] = (
        feat[["n_invoices_90d", "monetary_90d", "n_invoices_180d", "monetary_180d"]].fillna(0)
    )

    # Features that need two of the numbers above
    feat["recency_days"] = (cutoff - feat["last_purchase"]).dt.days
    feat["tenure_days"] = (cutoff - feat["first_purchase"]).dt.days
    feat["invoices_trend"] = feat["n_invoices_90d"] - (feat["n_invoices_180d"] - feat["n_invoices_90d"])
    feat["overdue_ratio"] = feat["recency_days"] / feat["avg_gap_days"]
    feat["return_rate"] = feat["return_value"] / feat["monetary_total"]

    # Keep only the final features, in a readable order
    feature_columns = [
        "recency_days", "n_invoices", "n_purchase_days", "n_invoices_90d", "n_invoices_180d", "invoices_trend",
        "monetary_total", "median_invoice_value", "monetary_90d", "monetary_180d", "tenure_days",
        "avg_gap_days", "overdue_ratio", "avg_items_per_invoice", "avg_lines_per_invoice", "n_products",
        "n_return_lines", "return_rate", "is_uk",
    ]
    feat = feat[feature_columns].reset_index()
    feat.insert(1, "cutoff", cutoff)
    return feat

In [7]:
# Try the function on ONE snapshot (the test snapshot) before building all of them
test_cutoff = pd.Timestamp("2011-09-10")
example = make_features(test_cutoff)

print("Rows:", len(example), "| Columns:", example.shape[1])
print("Same customers as in the labels:", set(example["customer_id"]) == set(labels.loc[labels["cutoff"] == test_cutoff, "customer_id"]))

# Which features have empty values?
print(example.isna().sum()[lambda s: s > 0])
example.head()

Rows: 2783 | Columns: 21
Same customers as in the labels: True
avg_gap_days     410
overdue_ratio    410
dtype: int64


,customer_id,cutoff,recency_days,n_invoices,n_purchase_days,n_invoices_90d,n_invoices_180d,invoices_trend,monetary_total,median_invoice_value,monetary_90d,monetary_180d,tenure_days,avg_gap_days,overdue_ratio,avg_items_per_invoice,avg_lines_per_invoice,n_products,n_return_lines,return_rate,is_uk
0,12347,2011-09-10,38,6,6,1.0,3,-1.0,3402.39,598.22,584.91,1603.68,313,55.000000,0.690909,349.833333,27.333333,107,0.0,0.000000,0
1,12348,2011-09-10,157,4,4,0.0,1,-1.0,1388.40,274.08,0.00,327.00,347,63.333333,2.478947,622.000000,11.000000,24,0.0,0.000000,0
2,12352,2011-09-10,171,6,6,0.0,2,-2.0,985.31,132.04,0.00,240.66,301,26.000000,6.576923,72.833333,8.500000,36,7.0,0.122124,0
3,12353,2011-09-10,113,2,2,0.0,1,-1.0,406.76,203.38,0.00,89.00,317,204.000000,0.553922,106.000000,12.000000,23,0.0,0.000000,0
4,12354,2011-09-10,141,1,1,0.0,1,-1.0,1079.40,1079.40,0.00,1079.40,141,NaN,NaN,530.000000,58.000000,58,0.0,0.000000,0


## Build the features of every snapshot

In [8]:
# Build the features of every snapshot and stack them
features = pd.concat([make_features(c) for c in sorted(labels["cutoff"].unique())], ignore_index=True)

# Count columns are whole numbers (they became decimals because empty values were filled with 0)
count_cols = ["n_invoices_90d", "n_invoices_180d", "invoices_trend", "n_return_lines"]
features[count_cols] = features[count_cols].astype(int)

# Add the label and the split, so every row has the features and the answer
data = features.merge(
    labels[["customer_id", "cutoff", "split", "churn"]],
    on=["customer_id", "cutoff"], how="left", validate="one_to_one",
)

print("Rows:", f"{len(data):,}", "| Rows in labels:", f"{len(labels):,}")
print("Rows without a label:", data["churn"].isna().sum())

Rows: 36,297 | Rows in labels: 36,297
Rows without a label: 0


In [9]:
# Empty values per feature (only the two gap features may have them)
print(data.isna().sum()[lambda s: s > 0])

# Basic logic checks: all three must be 0 or True
print("Rows with negative recency:", (data["recency_days"] < 0).sum())
print("Rows where 90-day invoices > 180-day invoices:", (data["n_invoices_90d"] > data["n_invoices_180d"]).sum())
print("Every scored customer has a purchase in the last 180 days:", (data["n_invoices_180d"] >= 1).all())

avg_gap_days     10205
overdue_ratio    10205
dtype: int64
Rows with negative recency: 0
Rows where 90-day invoices > 180-day invoices: 0
Every scored customer has a purchase in the last 180 days: True


In [12]:
# TIME TRAVEL TEST: build the features with ALL the data, then again after deleting everything from the
# cutoff date onwards. If a feature secretly uses the future, the two results will differ.
sales_all, returns_all, invoices_all = sales, returns, invoices
results = {}

try:
    for cutoff in sorted(labels["cutoff"].unique()):
        cutoff = pd.Timestamp(cutoff)

        # 1) Features built with ALL the data (restore the full tables first)
        sales, returns, invoices = sales_all, returns_all, invoices_all
        with_future = make_features(cutoff)

        # 2) Features built after deleting everything from the cutoff date onwards
        sales = sales_all[sales_all["invoicedate"] < cutoff]
        returns = returns_all[returns_all["invoicedate"] < cutoff]
        invoices = invoices_all[invoices_all["invoice_date"] < cutoff]
        without_future = make_features(cutoff)

        results[cutoff.date()] = with_future.equals(without_future)
finally:
    # Always put the full data back, even if something goes wrong
    sales, returns, invoices = sales_all, returns_all, invoices_all

print(pd.Series(results))
print("All snapshots identical:", all(results.values()))

2010-06-14    True
2010-07-14    True
2010-08-13    True
2010-09-12    True
2010-10-12    True
2010-11-11    True
2010-12-11    True
2011-01-10    True
2011-02-09    True
2011-03-11    True
2011-06-12    True
2011-09-10    True
dtype: bool
All snapshots identical: True


## Save the features

In [11]:
# Save features + label + split together, so the modeling notebook can start from here
data.to_parquet("../data/processed/features.parquet", index=False)
print("Saved features:", data.shape)

Saved features: (36297, 23)
